# テイラー展開 3：線形近似とヤコビ行列 ―$\det J=0$ で何が起きるか

多変数の写像 $\mathbf f:\mathbb R^2\to\mathbb R^2$ のテイラー展開の1次の項は、**ヤコビ行列** $J$ です：

$$
\mathbf f(\mathbf p+\mathbf h)=\mathbf f(\mathbf p)+J(\mathbf p)\,\mathbf h+O(|\mathbf h|^2),\qquad J_{ij}=\frac{\partial f_i}{\partial x_j}
$$

ヤコビ行列は、これまでのノート（計量テンソル、クリストッフェル記号、ラプラス・ベルトラミ、多様体）で何度も出てきました。このノートブックでは、

1. ヤコビ行列が1次の近似であること（誤差が $O(|\mathbf h|^2)$）
2. 逆写像のヤコビ行列は、ヤコビ行列の逆行列（逆関数定理）
3. **$\det J=0$ のとき、逆写像が作れなくなる3つの典型**（スライダーで点を動かす）
4. ニュートン法：1次の近似をくり返して方程式を解く

を見ます。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from math import factorial
from numpy.typing import NDArray
from plotly.subplots import make_subplots

Floats = NDArray[np.float64]       # 実数の配列（型ヒント用の別名）
Complexes = NDArray[np.complex128]
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなくリストを渡す（NB-02 を参照）

Vec = NDArray[np.float64]          # 2次元のベクトル


def jacobian_numeric(f, p: Vec, h: float = 1e-6) -> NDArray[np.float64]:
    """中心差分によるヤコビ行列（列が ∂f/∂x_j）"""
    cols = [(f(p + h * e) - f(p - h * e)) / (2 * h) for e in np.eye(len(p))]
    return np.stack(cols, axis=1)


def polar(p: Vec) -> Vec:
    r, th = p
    return np.array([r * np.cos(th), r * np.sin(th)])


def polar_J(p: Vec) -> NDArray[np.float64]:
    r, th = p
    return np.array([[np.cos(th), -r * np.sin(th)], [np.sin(th), r * np.cos(th)]])

## 1. ヤコビ行列は1次の近似

極座標 $(r,\theta)\mapsto(x,y)=(r\cos\theta,r\sin\theta)$ で、$\mathbf f(\mathbf p+\mathbf h)-\mathbf f(\mathbf p)-J\mathbf h$ の大きさを $|\mathbf h|$ に対して描きます。両対数で傾き 2（$O(|\mathbf h|^2)$）なら、$J$ が1次の近似の係数です。

In [ ]:
p0 = np.array([1.5, 0.6])
assert np.allclose(jacobian_numeric(polar, p0), polar_J(p0), atol=1e-8)
direction = np.array([0.6, -0.8])
hs = np.logspace(-5, -0.5, 15)
err = np.array([np.linalg.norm(polar(p0 + h * direction) - polar(p0) - polar_J(p0) @ (h * direction)) for h in hs])
slope = np.polyfit(np.log(hs), np.log(err), 1)[0]
print(f"両対数の傾き = {slope:.3f}（O(|h|²) なら 2）")
assert abs(slope - 2) < 0.1
fig = go.Figure([go.Scatter(x=hs.tolist(), y=err.tolist(), mode="lines+markers", name="|f(p+h) − f(p) − J h|")])
fig.update_layout(height=340, width=720, xaxis_type="log", yaxis_type="log", xaxis_title="|h|", margin=dict(t=20))
fig

## 2. 逆写像のヤコビ行列は、ヤコビ行列の逆行列

$\det J(\mathbf p)\ne0$ なら、$\mathbf p$ の近くで逆写像 $\mathbf f^{-1}$ があり、そのヤコビ行列は $J(\mathbf p)^{-1}$ です（**逆関数定理**。[関数・線形写像・微分・積分のノート](../../../研究ノート/01_基礎・ベクトル解析/functions_linear_maps_derivatives_integrals.md)の Part III §9、証明は付録B）。極座標の逆写像 $(x,y)\mapsto(r,\theta)=(\sqrt{x^2+y^2},\operatorname{atan2}(y,x))$ で確かめます。極座標では $\det J=r$ です。

In [ ]:
def polar_inv(q: Vec) -> Vec:
    return np.array([np.hypot(q[0], q[1]), np.arctan2(q[1], q[0])])


for p in (np.array([1.5, 0.6]), np.array([0.3, 2.5]), np.array([2.0, -1.0])):
    J_f = polar_J(p)
    J_inv_map = jacobian_numeric(polar_inv, polar(p))
    assert np.isclose(np.linalg.det(J_f), p[0])                                # det J = r
    assert np.allclose(J_inv_map, np.linalg.inv(J_f), atol=1e-6)               # 逆写像のヤコビ行列 = J⁻¹
    print(f"r = {p[0]}, θ = {p[1]}：det J = {np.linalg.det(J_f):.3f}、逆写像のヤコビ行列 = J⁻¹ を確認")

## 3. $\det J=0$ で何が起きるか：3つの典型

**ヤコビ行列は、点ごとの「局所的なゆがみ」**：ある行列の行列式は1つに決まりますが、ヤコビ行列 $J(\mathbf p)$ は**点 $\mathbf p$ ごとに違う行列**です。たとえば極座標では

$$
J(r,\theta)=\begin{pmatrix}\cos\theta&-r\sin\theta\\ \sin\theta&r\cos\theta\end{pmatrix},\qquad \det J(r,\theta)=r
$$

で、成分が $r,\theta$ によって変わります。$\mathbf p$ のごく近くだけを見ると、写像は線形写像 $J(\mathbf p)$ とみなせる（§1）ので、

- $\mathbf p$ のまわりの小さな円は、$J(\mathbf p)$ で決まる**楕円**に写る（どの向きに何倍伸びるか、どれだけ回るか）
- その面積の比が $|\det J(\mathbf p)|$、符号が向き（裏返るかどうか）

です。つまり $\det J(\mathbf p)$ は「点 $\mathbf p$ のまわりで、面積が何倍になるか」を表す、**点の関数**です。線形写像なら $J$ はどこでも同じ行列なので $\det J$ は定数ですが、非線形な写像では場所によって変わり、「$\det J(\mathbf p)=0$ となる点の集合」が、線になったり、平面全体になったりします。

$\det J(\mathbf p)=0$ の点では逆行列が作れず、逆関数定理が使えません。そこでは、写像は次のどれかの形で「壊れて」います。

| 写像 | $\det J(\mathbf p)$ | 壊れ方 |
|---|---|---|
| 極座標 $(r,\theta)\mapsto(r\cos\theta,r\sin\theta)$ | $r$ | $r=0$ の線全体が原点1点に**つぶれる**（座標特異点。[多様体のノート](../../../研究ノート/03_多様体・微分形式・トポロジー/manifolds_introduction.md)の「極は座標特異点」） |
| 折り返し $(x,y)\mapsto(x^2,y)$ | $2x$ | $x=0$ を境に**折り返し**、左右の2点が同じ点に写る（2対1） |
| 一方向につぶす $(x,y)\mapsto(x+y,\ 2x+2y)$ | $0$（どこでも） | 平面全体が1本の直線に**つぶれる** |

左の図は元の平面（格子と、点 $\mathbf p$ のまわりの小さな円）、右の図は写した先です。格子の点は $\det J$ の符号で色分けしています（青：正、赤：負、黒：ほぼ $0$）。点 $\mathbf p$ のスライダーを動かして、小さな円が写った先（$J(\mathbf p)$ で決まる楕円）がつぶれる様子を見ます。円と像の面積の比も表示するので、$|\det J(\mathbf p)|$ と比べてください（円が小さいほどよく一致します）。

In [ ]:
def fold(p: Vec) -> Vec:
    return np.array([p[0] ** 2, p[1]])


def squash(p: Vec) -> Vec:
    return np.array([p[0] + p[1], 2 * p[0] + 2 * p[1]])


MAPS = {
    "polar": ("極座標 (r, θ) → (r cos θ, r sin θ)", polar, (0.0, 2.0), (0.0, 2 * np.pi)),
    "fold": ("折り返し (x, y) → (x², y)", fold, (-1.5, 1.5), (-1.5, 1.5)),
    "squash": ("一方向につぶす (x, y) → (x + y, 2x + 2y)", squash, (-1.5, 1.5), (-1.5, 1.5)),
}
assert np.isclose(np.linalg.det(jacobian_numeric(fold, np.array([0.7, 0.2]))), 1.4)
assert abs(np.linalg.det(jacobian_numeric(squash, np.array([0.3, -0.4])))) < 1e-9


def grid_lines(f, xr: tuple[float, float], yr: tuple[float, float], k: int = 13) -> tuple[list, list, list, list]:
    """元の平面の格子線と、写した先の格子線（None で区切った座標の列）"""
    dx, dy, ix, iy = [], [], [], []
    for a in np.linspace(*xr, k):
        ys = np.linspace(*yr, 60)
        pts = np.array([[a, b] for b in ys])
        img = np.array([f(q) for q in pts])
        dx += pts[:, 0].tolist() + [None]; dy += pts[:, 1].tolist() + [None]
        ix += img[:, 0].tolist() + [None]; iy += img[:, 1].tolist() + [None]
    for b in np.linspace(*yr, k):
        xs = np.linspace(*xr, 60)
        pts = np.array([[a, b] for a in xs])
        img = np.array([f(q) for q in pts])
        dx += pts[:, 0].tolist() + [None]; dy += pts[:, 1].tolist() + [None]
        ix += img[:, 0].tolist() + [None]; iy += img[:, 1].tolist() + [None]
    return dx, dy, ix, iy


fig_jac = go.FigureWidget(make_subplots(rows=1, cols=2, subplot_titles=("元の平面", "写した先")))
for col in (1, 2):
    fig_jac.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#ccc", width=1), hoverinfo="skip", showlegend=False), 1, col)
    fig_jac.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(size=5), hoverinfo="skip", showlegend=False), 1, col)
    fig_jac.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#e67e00", width=3), showlegend=False), 1, col)
    fig_jac.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(size=9, color="#222"), showlegend=False), 1, col)
fig_jac.update_layout(height=480, width=980, margin=dict(t=40, l=40, r=20, b=40))
map_dd = W.Dropdown(options=[(v[0], k) for k, v in MAPS.items()], value="polar", description="写像", layout=W.Layout(width="420px"))
px_s = W.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.01, description="p の位置 1")
py_s = W.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01, description="p の位置 2")
info3 = W.HTML()


def update_jac(*_: object) -> None:
    """写像と点 p が変わるたびに、格子・小さな円・ヤコビ行列を描き直す"""
    name, f, xr, yr = MAPS[map_dd.value]
    p = np.array([xr[0] + px_s.value * (xr[1] - xr[0]), yr[0] + py_s.value * (yr[1] - yr[0])])
    dx, dy, ix, iy = grid_lines(f, xr, yr)
    gx, gy = np.meshgrid(np.linspace(*xr, 17), np.linspace(*yr, 17))
    pts = np.stack([gx.ravel(), gy.ravel()], axis=1)
    dets = np.array([np.linalg.det(jacobian_numeric(f, q)) for q in pts])
    colors = ["#222" if abs(d) < 1e-6 else ("#1f77b4" if d > 0 else "#d62728") for d in dets]
    img_pts = np.array([f(q) for q in pts])
    rad = 0.12 * (xr[1] - xr[0])
    ang = np.linspace(0, 2 * np.pi, 80)
    circle = np.stack([p[0] + rad * np.cos(ang), p[1] + rad * np.sin(ang)], axis=1)
    circle_img = np.array([f(q) for q in circle])
    J_p = jacobian_numeric(f, p)
    det_p = float(np.linalg.det(J_p))
    area = 0.5 * abs(np.dot(circle_img[:, 0], np.roll(circle_img[:, 1], -1)) - np.dot(circle_img[:, 1], np.roll(circle_img[:, 0], -1)))
    ratio = area / (np.pi * rad**2)                     # 像の面積 ÷ 円の面積（靴ひも公式）
    with fig_jac.batch_update():
        fig_jac.data[0].x, fig_jac.data[0].y = dx, dy
        fig_jac.data[1].x, fig_jac.data[1].y = pts[:, 0].tolist(), pts[:, 1].tolist()
        fig_jac.data[1].marker.color = colors
        fig_jac.data[2].x, fig_jac.data[2].y = circle[:, 0].tolist(), circle[:, 1].tolist()
        fig_jac.data[3].x, fig_jac.data[3].y = [float(p[0])], [float(p[1])]
        fig_jac.data[4].x, fig_jac.data[4].y = ix, iy
        fig_jac.data[5].x, fig_jac.data[5].y = img_pts[:, 0].tolist(), img_pts[:, 1].tolist()
        fig_jac.data[5].marker.color = colors
        fig_jac.data[6].x, fig_jac.data[6].y = circle_img[:, 0].tolist(), circle_img[:, 1].tolist()
        fq = f(p)
        fig_jac.data[7].x, fig_jac.data[7].y = [float(fq[0])], [float(fq[1])]
    Jtxt = f"[[{J_p[0, 0]:+.2f}, {J_p[0, 1]:+.2f}], [{J_p[1, 0]:+.2f}, {J_p[1, 1]:+.2f}]]"
    if abs(det_p) > 1e-6:
        Ji = np.linalg.inv(J_p)
        inv_txt = f"逆行列 J⁻¹ = [[{Ji[0, 0]:+.2f}, {Ji[0, 1]:+.2f}], [{Ji[1, 0]:+.2f}, {Ji[1, 1]:+.2f}]] → p の近くに逆写像がある"
    else:
        inv_txt = "<b>det J = 0：逆行列が作れない</b>（小さな円が線分につぶれる。p の近くで逆写像が作れない）"
    info3.value = (f"p = ({p[0]:.2f}, {p[1]:.2f})：J(p) = {Jtxt}、det J(p) = {det_p:+.3f}、"
                   f"面積の比（像 ÷ 円）≈ {ratio:.3f}<br>{inv_txt}")


for w in (map_dd, px_s, py_s):
    w.observe(update_jac, "value")
update_jac()
W.VBox([map_dd, W.HBox([px_s, py_s]), info3, fig_jac])

**考えてみよう**

- **極座標**：「p の位置 1」（$r$）を $0$ に近づけると、小さな円の像が細長くつぶれていき、$r=0$ で線分になります。$\theta$ がいくつでも、$r=0$ なら原点です。逆写像（角度 $\theta$ を求める）が決まらなくなるのは、そのためです。
- **折り返し**：$x$ を $0$ に近づけると、像の楕円が横につぶれ、$x=0$ で線分になります。$x<0$（赤）と $x>0$（青）の格子が、右の図で重なっている（2対1）ことも確かめましょう。行列式の符号が変わるのは、向きが反転するからです。
- **一方向につぶす**：どこに $\mathbf p$ を置いても、像は線分です。$J$ が定数行列（線形写像）なので、$\det J(\mathbf p)=0$ が一部の線の上ではなく、平面全体で起きている場合です。

## 4. ニュートン法：1次の近似をくり返す

方程式 $\mathbf f(\mathbf x)=\mathbf y$ を解くために、今の推定 $\mathbf x_k$ のまわりで1次の近似 $\mathbf f(\mathbf x_k)+J(\mathbf x_k)(\mathbf x-\mathbf x_k)=\mathbf y$ を解き、それを次の推定にします：

$$
\mathbf x_{k+1}=\mathbf x_k-J(\mathbf x_k)^{-1}\big(\mathbf f(\mathbf x_k)-\mathbf y\big)
$$

1次の近似の誤差は $O(|\mathbf h|^2)$ なので、解の近くでは**誤差が1回ごとにおよそ2乗になる**（桁数が倍々に増える）速さで収束します。ただし $J$ の逆行列を使うので、$\det J$ が $0$ に近い所から始めると、1歩が大きく飛んでしまいます。極座標で $(x,y)=(1,1)$ になる $(r,\theta)$（答えは $(\sqrt2,\pi/4)$）を探します。

In [ ]:
def newton(f, Jf, y: Vec, x0: Vec, steps: int = 8) -> list[Vec]:
    xs = [x0]
    for _ in range(steps):
        x = xs[-1]
        xs.append(x - np.linalg.solve(Jf(x), f(x) - y))
    return xs


target = np.array([1.0, 1.0])
answer = np.array([np.sqrt(2), np.pi / 4])
good = newton(polar, polar_J, target, np.array([1.0, 0.5]))
errs = [float(np.linalg.norm(x - answer)) for x in good]
print("良い出発点 (1.0, 0.5) からの誤差：", ["%.1e" % e for e in errs[:6]])
for k in range(1, 4):                                   # 誤差がおよそ 2 乗になる
    assert errs[k + 1] < 2 * errs[k] ** 2 + 1e-14
bad = newton(polar, polar_J, target, np.array([0.02, 0.5]), steps=3)
print("det J ≈ 0（r = 0.02）から始めた最初の1歩：", np.round(bad[0], 3), "→", np.round(bad[1], 2),
      f"（det J = {np.linalg.det(polar_J(bad[0])):.2f} なので J⁻¹ が大きく、遠くへ飛ぶ）")
assert np.linalg.norm(bad[1] - bad[0]) > 10

## 観察のポイント（まとめ）

- ヤコビ行列は、多変数のテイラー展開の1次の項（最良の線形近似）で、誤差は $O(|\mathbf h|^2)$。点ごとに違う行列で、その点のまわりの「局所的なゆがみ」（小さな円 → 楕円）を表し、$|\det J(\mathbf p)|$ は面積の倍率。
- $\det J\ne0$ なら、その点の近くで逆写像があり、そのヤコビ行列は $J^{-1}$（逆関数定理）。
- $\det J=0$ の点では、写像が「つぶれる」か「折り返す」かしていて、逆写像が作れない。極座標の原点（座標特異点）は、その典型。
- ニュートン法は1次の近似のくり返しで、解の近くでは誤差がおよそ2乗ずつ減るが、$\det J\approx0$ の所では大きく飛ぶ。
- カオスのノートブック（[ローレンツ系](../../learning/14_chaos/02_lorenz.ipynb)）でも、近い軌道の差の広がりを、ヤコビ行列（1次の近似）で調べました。